# Demo — Respostas prontas: o FAQ aprovado antes da pergunta

**Objetivo:** reproduzir a linha **Respostas prontas** do slide 18 ("RAGless: cada variante tira uma peça do pipeline clássico"): na ingestão, o LLM gera pares pergunta-resposta; na consulta, a busca vetorial devolve a resposta pronta. É o FAQ fechado e auditável de regulamento, política e atendimento.

1. **Ingestão** → o LLM lê três políticas e escreve pares pergunta-resposta, cada um com o trecho que o sustenta;
2. **Curadoria auditável** → os pares aprovados vão para um arquivo, que é o artefato de auditoria;
3. **Vetorização** → só as perguntas aprovadas viram vetor;
4. **Consulta sem LLM** → a paráfrase recebe a resposta pronta, e o que está fora do FAQ é encaminhado;
5. **Custo e previsibilidade** → um embedding por pergunta, nenhum token de geração e sempre a mesma resposta.

> **Cenário 100% fictício.** A Contoso Industrial e todos os documentos, projetos, pessoas e números do `acervo/` existem apenas para fins didáticos.

**O que sai do pipeline clássico nesta variante:** o LLM na hora da consulta. A busca vetorial continua, mas compara a pergunta do usuário com perguntas já aprovadas e devolve uma resposta escrita antes, em vez de entregar trechos para o modelo redigir. Se o slide 18 ainda disser “Sem vetor” nesta linha, leia pela descrição da própria linha, que usa a busca vetorial na consulta: o rótulo foi trocado com o da **Árvore (PageIndex)**.

**Sem Azure AI Search neste exemplo:** os pares aprovados ficam num arquivo JSON, e a comparação vetorial é um cosseno calculado com numpy. Em produção, eles morariam num índice (no AI Search, por exemplo), e a consulta continuaria sem chamar um LLM.

## 0. Preparação

Este notebook usa a **Azure OpenAI v1 API** (cliente Python `OpenAI`): o deployment de chat só na ingestão, para escrever os pares, e o de embedding na ingestão e na consulta.

Variáveis necessárias:

- `AZURE_OPENAI_ENDPOINT` e `AZURE_OPENAI_API_KEY`
- `AZURE_OPENAI_DEPLOYMENT` (deployment de chat com suporte a structured outputs, como `gpt-5-mini`; usado só na Execução 1)
- `AZURE_OPENAI_EMBEDDING_DEPLOYMENT`, `AZURE_OPENAI_EMBEDDING_MODEL` e `AZURE_OPENAI_EMBEDDING_DIMENSIONS`

Os valores vêm do arquivo `Cap05/.env` (copie o `.env.example` e preencha); o que faltar é perguntado na hora da execução. Este notebook não usa o Azure AI Search nem o índice de `tipos-de-rag`.

> Não grave chaves diretamente no notebook.

In [ ]:
%pip install -q "openai>=2" "python-dotenv>=1.0" "pandas>=2.1" "numpy>=1.26" "pydantic>=2.7"

In [ ]:
import os
from getpass import getpass
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

load_dotenv(find_dotenv(usecwd=True))  # lê o .env da pasta Cap05, se existir

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT") or input(
    "AZURE_OPENAI_ENDPOINT (ex.: https://meu-recurso.openai.azure.com): "
).strip()
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY") or getpass("AZURE_OPENAI_API_KEY: ")
AZURE_OPENAI_DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT") or input(
    "AZURE_OPENAI_DEPLOYMENT (deployment de chat): "
).strip()
AZURE_OPENAI_EMBEDDING_DEPLOYMENT = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT") or input(
    "AZURE_OPENAI_EMBEDDING_DEPLOYMENT (deployment de embedding): "
).strip()

# O modelo e a dimensão do embedding são o contrato entre indexação e consulta.
EMBEDDING_MODEL = os.getenv("AZURE_OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")
EMBEDDING_DIMENSIONS = int(os.getenv("AZURE_OPENAI_EMBEDDING_DIMENSIONS", "1536"))

client = OpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    base_url=f"{AZURE_OPENAI_ENDPOINT.rstrip('/')}/openai/v1/",
)

# Pasta Cap05: a primeira pasta, subindo a partir daqui, que contém o acervo.
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "acervo").is_dir())
PASTA_ACERVO = RAIZ / "acervo"

print("Cliente configurado.")
print("Deployment de chat:", AZURE_OPENAI_DEPLOYMENT)
print("Embedding:", AZURE_OPENAI_EMBEDDING_DEPLOYMENT, f"({EMBEDDING_MODEL}, {EMBEDDING_DIMENSIONS} dimensões)")
print("Acervo:", PASTA_ACERVO)

### Funções auxiliares

- `carregar_trechos`: a mesma dos outros notebooks do Cap. 5 que a usam, um trecho por seção `## `. Aqui ela entrega ao LLM as seções das três políticas, cada uma com o seu `id`.
- `gerar_embeddings`: a mesma dos outros notebooks que geram embeddings. Vetoriza as perguntas aprovadas (Execução 3) e a pergunta do usuário (Execução 4), com o mesmo deployment e a mesma dimensão.

In [ ]:
import re

pd.set_option("display.max_colwidth", None)  # tabelas mostram o texto inteiro


def carregar_trechos(pasta_acervo: Path) -> list[dict]:
    """Lê os .md do acervo e devolve um trecho por seção "## " (chunking pela estrutura do documento)."""
    trechos = []
    for caminho in sorted(pasta_acervo.rglob("*.md")):
        fonte = caminho.relative_to(pasta_acervo).as_posix()
        doc_id = fonte.removesuffix(".md").replace("/", "_")
        texto = caminho.read_text(encoding="utf-8")
        titulo = texto.splitlines()[0].lstrip("# ").strip()
        # O que vem antes do primeiro "## " é o cabeçalho do documento (título e metadados).
        for numero, secao in enumerate(re.split(r"(?m)^## ", texto)[1:], start=1):
            nome_secao, _, conteudo = secao.partition("\n")
            trechos.append({
                "id": f"{doc_id}_{numero:02d}",
                "doc_id": doc_id,
                "titulo": titulo,
                "secao": nome_secao.strip(),
                "conteudo": conteudo.strip(),
                "fonte": fonte,
            })
    return trechos


def gerar_embeddings(textos: list[str], lote: int = 16) -> list[list[float]]:
    vetores = []
    for i in range(0, len(textos), lote):
        resposta = client.embeddings.create(
            model=AZURE_OPENAI_EMBEDDING_DEPLOYMENT,
            input=textos[i:i + lote],
            dimensions=EMBEDDING_DIMENSIONS,
        )
        vetores.extend(item.embedding for item in resposta.data)
    return vetores


trechos = carregar_trechos(PASTA_ACERVO)
print(f"{len(trechos)} trechos de {len({t['doc_id'] for t in trechos})} documentos")

---

# Execução 1 — Ingestão: o LLM escreve o FAQ

Três políticas com cara de FAQ vão para o modelo, **uma chamada por documento**: descanso remunerado (as férias), viagens e reembolso, e cartão corporativo. Cada chamada recebe as seções do documento, cada uma com o seu `id`, e devolve de 4 a 6 pares num formato fixo (**structured outputs**, a partir de um modelo Pydantic): `pergunta`, `resposta` e `trecho_id`, o trecho que sustenta a resposta.

É aqui, e só aqui, que o LLM trabalha: a geração é paga uma vez por versão da política, e não a cada pergunta.

### O que esperar e por que pode variar

- Cada execução gera perguntas diferentes: o modelo escolhe o que perguntar e como redigir. É por isso que a consulta não usa a geração, e sim o arquivo aprovado (Execução 2).
- O structured outputs garante o formato, não o conteúdo. Um par que cita um `trecho_id` que não foi enviado é descartado aqui; o resto passa pela revisão humana.

In [ ]:
from pydantic import BaseModel


# O formato que o modelo é obrigado a seguir (structured outputs).
class Par(BaseModel):
    pergunta: str
    resposta: str
    trecho_id: str


class Pares(BaseModel):
    pares: list[Par]


# O esquema fixa o formato, mas não aceita minItems/maxItems: a quantidade de pares vai nas instruções.
INSTRUCOES_INGESTAO = """
Você escreve o FAQ oficial de uma política interna, a partir dos trechos fornecidos.
Escreva, em português, de 4 a 6 pares de pergunta e resposta, com as dúvidas mais comuns dos colaboradores.
- pergunta: como o colaborador perguntaria no dia a dia, com as palavras dele, e não com os termos do documento.
- resposta: curta e completa, usando só o que está nos trechos, sem inventar regras, valores ou exceções.
- trecho_id: o id do trecho que sustenta a resposta, copiado sem os colchetes.
""".strip()


def gerar_pares(trechos_do_documento: list[dict]):
    """Uma chamada ao LLM por documento: entram as seções com id, saem os pares no formato de Pares."""
    contexto = "\n\n".join(
        f"[{t['id']}] {t['titulo']} — {t['secao']}\n{t['conteudo']}" for t in trechos_do_documento
    )
    resposta = client.responses.parse(
        model=AZURE_OPENAI_DEPLOYMENT,
        instructions=INSTRUCOES_INGESTAO,
        input=f"TRECHOS DO DOCUMENTO:\n{contexto}",
        text_format=Pares,
        store=False,
    )
    if resposta.output_parsed is None:
        raise RuntimeError("O modelo não devolveu os pares no formato pedido (por exemplo, recusou o pedido).")
    return resposta


DOCUMENTOS_FAQ = [
    "rh/descanso-remunerado-anual.md",
    "financeiro/reembolso-de-despesas-de-viagem.md",
    "financeiro/cartao-corporativo.md",
]

pares_gerados, uso_ingestao = [], []
for fonte in DOCUMENTOS_FAQ:
    trechos_do_documento = [t for t in trechos if t["fonte"] == fonte]
    resposta = gerar_pares(trechos_do_documento)
    pares = resposta.output_parsed.pares
    # O formato é garantido; o conteúdo, não: descartamos o par que cita um trecho que não foi enviado.
    ids_enviados = {t["id"] for t in trechos_do_documento}
    validos = [par for par in pares if par.trecho_id in ids_enviados]
    pares_gerados += [
        {"pergunta": par.pergunta, "resposta": par.resposta, "trecho_id": par.trecho_id, "fonte": fonte}
        for par in validos
    ]
    uso_ingestao.append({
        "documento": fonte,
        "pares gerados": len(pares),
        "pares com trecho válido": len(validos),
        "tokens de entrada": resposta.usage.input_tokens,
        "tokens de saída": resposta.usage.output_tokens,
    })

display(pd.DataFrame(uso_ingestao))
if not pares_gerados:  # não deixa gravar uma curadoria vazia na Execução 2
    raise RuntimeError("Nenhum par citou um trecho válido: confira o trecho_id que o modelo devolveu.")
print(f"{len(pares_gerados)} pares prontos para a curadoria.")

### O que observar

- Cada par aponta o trecho de origem. A rastreabilidade que o auditor pede é decidida aqui, na ingestão.
- A coluna **pares com trecho válido** mostra quantos pares passaram na conferência do `trecho_id`.
- Os tokens da tabela são todo o custo de geração deste FAQ. Em produção, eles só voltam a ser pagos quando uma política mudar; aqui o notebook gera de novo a cada execução só para mostrar a ingestão.

---

# Execução 2 — Curadoria auditável

Em produção, **uma pessoa da área dona da política revisa cada par** antes de ele ir para o ar: confere a resposta com o documento, ajusta a redação e aprova. O arquivo `respostas_aprovadas.json`, gravado ao lado deste notebook, é o artefato de auditoria: diz o que o sistema pode responder, com que palavras e com base em que trecho. Quando a política muda, os pares daquele documento são gerados e aprovados de novo.

Em sala, simulamos a revisão aprovando todos os pares (`"aprovada": true`). Se o arquivo já existir, por exemplo numa segunda execução, ele é carregado e **vale mais que a geração de agora**: o que foi aprovado não muda sozinho. É também o que permite seguir daqui se a Execução 1 falhar no dia: a consulta não depende do LLM.

> Para ver a curadoria em ação, mude um `"aprovada"` para `false` no arquivo e rode o notebook de novo a partir desta seção: aquele par sai da consulta. Na Execução 4, a pergunta parecida com ele passa a ser encaminhada ou, se outra pergunta aprovada passar do limiar, recebe a resposta dela, que é o erro perigoso descrito lá. Para refazer a curadoria, apague o arquivo.

In [ ]:
import json

ARQUIVO_APROVADAS = Path.cwd() / "respostas_aprovadas.json"  # ao lado do notebook

if ARQUIVO_APROVADAS.exists():
    curadoria = json.loads(ARQUIVO_APROVADAS.read_text(encoding="utf-8"))
    print(f"Curadoria carregada de {ARQUIVO_APROVADAS}. Ela vale mais que a geração de agora.")
elif len(uso_ingestao) < len(DOCUMENTOS_FAQ) or not pares_gerados:  # não grava um FAQ pela metade
    raise RuntimeError("A Execução 1 não terminou com pares válidos: rode-a de novo antes da curadoria.")
else:
    # Em sala, a revisão é simulada: todos os pares saem aprovados. Em produção, uma pessoa aprova par a par.
    curadoria = [{**par, "aprovada": True} for par in pares_gerados]
    ARQUIVO_APROVADAS.write_text(json.dumps(curadoria, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"Curadoria gravada em {ARQUIVO_APROVADAS}")

aprovadas = [par for par in curadoria if par["aprovada"] is True]  # só o true do JSON aprova
if not aprovadas:
    raise RuntimeError('Nenhum par com "aprovada": true no arquivo: aprove ao menos um par ou apague o arquivo para refazer a curadoria.')
fora_da_curadoria = {p["pergunta"] for p in pares_gerados} - {p["pergunta"] for p in curadoria}
print(f"Pares aprovados: {len(aprovadas)} de {len(curadoria)}")
print(f"Perguntas geradas agora que não estão na curadoria: {len(fora_da_curadoria)} de {len(pares_gerados)}")

display(pd.DataFrame(curadoria)[["aprovada", "pergunta", "resposta", "trecho_id"]])

### Leitura arquitetural

- O LLM propõe, uma pessoa aprova. A resposta que chega ao usuário é a que alguém revisou, palavra por palavra.
- Reprovar um par tira a resposta dele do ar, mas não bloqueia o assunto: a pergunta parecida vai para a pergunta aprovada mais próxima. Para fechar um assunto, mantenha a pergunta e aprove como resposta um encaminhamento.
- Numa segunda execução, a linha **perguntas geradas agora que não estão na curadoria** mostra que o modelo não escreve o mesmo FAQ duas vezes. Por isso o que vale é o arquivo, e não a geração.
- Em produção, cada registro guarda também quem aprovou, quando e a versão do documento de origem: é o que diz quais respostas revisar quando a política muda.

---

# Execução 3 — Vetorizar as perguntas aprovadas

Só as **perguntas aprovadas** viram vetor, com o mesmo `gerar_embeddings` (mesmo deployment, mesma dimensão) que vai vetorizar a pergunta do usuário. As respostas não viram vetor: elas não são buscadas, são devolvidas.

Os vetores ficam numa matriz numpy, com cada linha normalizada para tamanho 1. Assim, o produto escalar entre dois vetores é o cosseno entre eles. Em produção, essa matriz seria um índice vetorial.

In [ ]:
import numpy as np

matriz = np.array(gerar_embeddings([par["pergunta"] for par in aprovadas]))
matriz = matriz / np.linalg.norm(matriz, axis=1, keepdims=True)  # cada linha com tamanho 1

print(f"Matriz: {matriz.shape[0]} perguntas aprovadas x {matriz.shape[1]} dimensões")
print("Tamanho das três primeiras linhas:", np.linalg.norm(matriz[:3], axis=1).round(6))

---

# Execução 4 — Consulta sem LLM

`responder_pronto` faz **uma chamada de embedding** com a pergunta do usuário, calcula o cosseno com todas as perguntas aprovadas e pega a mais parecida. Se a similaridade alcançar o `LIMIAR`, devolve a resposta aprovada e a fonte; se não, encaminha para o atendimento. Nenhum modelo de linguagem é chamado.

Cinco perguntas de usuário: três paráfrases de assuntos que o FAQ cobre e duas fora do escopo.

### O que esperar e por que pode variar

- As paráfrases tendem a ficar acima do limiar, e as perguntas fora do escopo, abaixo. Mas a similaridade depende de como o modelo redigiu as perguntas aprovadas e do modelo de embedding: olhe os números, não só o resultado.
- Uma paráfrase legítima abaixo do limiar é encaminhada. O erro é seguro, mas mostra que faltou uma pergunta aprovada com aquelas palavras, ou que o limiar está alto.
- O erro perigoso é o contrário: uma pergunta fora do escopo acima do limiar recebe a resposta de outra pergunta. Repare em "Como peço um notebook novo?": ela tem a forma de várias perguntas do FAQ, um pedido de procedimento, mas fala de outro assunto.

In [ ]:
LIMIAR = 0.6  # ponto de partida, não verdade: calibre com perguntas reais dos usuários
ENCAMINHAMENTO = "Não tenho uma resposta aprovada para isso. Vou encaminhar ao atendimento."


def responder_pronto(pergunta: str) -> dict:
    """Consulta sem LLM: um embedding, o cosseno com cada pergunta aprovada e a resposta pronta ou o encaminhamento."""
    vetor = np.array(gerar_embeddings([pergunta])[0])
    similaridades = matriz @ (vetor / np.linalg.norm(vetor))
    melhor = int(similaridades.argmax())
    par = aprovadas[melhor]
    respondida = similaridades[melhor] >= LIMIAR
    return {
        "pergunta_aprovada": par["pergunta"],
        "similaridade": float(similaridades[melhor]),
        "resposta": par["resposta"] if respondida else ENCAMINHAMENTO,
        "fonte": f"{par['fonte']} ({par['trecho_id']})" if respondida else "",
    }


PERGUNTAS_DO_USUARIO = [
    "Dá para vender dez dias de férias?",
    "Qual o limite da diária de hotel em Manaus?",
    "Posso comprar passagem aérea com o cartão corporativo?",
    "Qual o horário do refeitório?",
    "Como peço um notebook novo?",
]

linhas = []
for pergunta in PERGUNTAS_DO_USUARIO:
    r = responder_pronto(pergunta)
    linhas.append({
        "pergunta do usuário": pergunta,
        "pergunta aprovada mais próxima": r["pergunta_aprovada"],
        "similaridade": round(r["similaridade"], 3),
        "resposta ou encaminhamento": r["resposta"],
        "fonte": r["fonte"],
    })

display(Markdown(f"**Limiar:** {LIMIAR}"))
display(pd.DataFrame(linhas))

### O que observar

- Compare a coluna **similaridade** com o `LIMIAR`: é só esse número que separa responder de encaminhar.
- Nas perguntas respondidas, a resposta é o texto aprovado, palavra por palavra, e a fonte traz o documento e o trecho que a sustentam, os mesmos do arquivo de curadoria.
- Nas encaminhadas, a **pergunta aprovada mais próxima** mostra o que o sistema quase respondeu. É o material para calibrar o limiar e para descobrir perguntas que o FAQ ainda não tem.

### Leitura arquitetural

- Em produção, com as perguntas aprovadas num índice do AI Search, a comparação com o limiar continua na aplicação, na pilha usada no capítulo (`azure-search-documents` 12.0.0 GA, API REST `2026-04-01`): nessa versão, a consulta vetorial não aceita limiar e sempre devolve os `k` vizinhos mais próximos. O Learn documenta um limiar de similaridade na própria consulta vetorial (`threshold`, do tipo `vectorSimilarity`), mas só em preview (API `2026-08-01-preview`). Atenção à escala: numa consulta vetorial com cosseno, o `@search.score` não é o cosseno, é `1 / (1 + distância de cosseno)`. Converta o score antes de comparar, ou calibre o limiar já na escala do score.

### Fala sugerida

> “Quando a pergunta não está no FAQ aprovado, o sistema não improvisa: encaminha.”

---

# Execução 5 — Custo e previsibilidade

A mesma pergunta, três vezes: a primeira pergunta do usuário que a Execução 4 respondeu com a resposta aprovada. Se nenhuma passou do limiar, a célula avisa e usa a primeira pergunta aprovada, como está no arquivo. No Cap. 1, a mesma pergunta deu três respostas diferentes; aqui, a resposta não é gerada na consulta, é lida do arquivo aprovado. Depois, a conta: o que se paga uma vez, na ingestão, e o que se paga a cada pergunta.

In [ ]:
# A primeira pergunta que a Execução 4 respondeu com a resposta aprovada: só as respondidas têm fonte.
respondidas = [linha["pergunta do usuário"] for linha in linhas if linha["fonte"]]
if respondidas:
    pergunta = respondidas[0]
else:
    pergunta = aprovadas[0]["pergunta"]
    print("Nenhuma pergunta do usuário passou do limiar na Execução 4. Usamos a primeira pergunta aprovada, como está no arquivo.")
repeticoes = [responder_pronto(pergunta) for _ in range(3)]

display(Markdown(f"**A mesma pergunta, três vezes:** {pergunta}"))
display(pd.DataFrame([
    {"execução": i, "similaridade": round(r["similaridade"], 6), "resposta": r["resposta"]}
    for i, r in enumerate(repeticoes, start=1)
]))
print("Respostas diferentes entre as três execuções:", len({r["resposta"] for r in repeticoes}))

tokens_ingestao = sum(u["tokens de entrada"] + u["tokens de saída"] for u in uso_ingestao)
display(pd.DataFrame([
    {
        "momento": "Ingestão: uma vez por versão das políticas",
        "chamadas ao LLM": len(uso_ingestao),
        "tokens do LLM (entrada + saída)": tokens_ingestao,
        "embeddings": f"as {len(aprovadas)} perguntas aprovadas, em lote",
        "de onde vem a resposta": "escrita pelo LLM e aprovada por uma pessoa",
    },
    {
        "momento": "Consulta: a cada pergunta do usuário",
        "chamadas ao LLM": 0,
        "tokens do LLM (entrada + saída)": 0,
        "embeddings": "1, só com a pergunta do usuário",
        "de onde vem a resposta": "lida do arquivo aprovado",
    },
]).set_index("momento"))

### O que observar

- A coluna **resposta** se repete nas três execuções. Se a similaridade variar nas últimas casas decimais, a resposta continua a mesma, porque ela não é gerada.
- Na consulta, o custo é uma chamada de embedding com uma frase curta, e nenhum token de geração. O LLM foi pago uma vez, na ingestão.

### Leitura arquitetural

- **Troca flexibilidade por previsibilidade.** Toda resposta foi revisada antes de existir a pergunta; em compensação, o sistema só responde o que foi aprovado.
- **O limiar é uma decisão de produto.** Alto demais, encaminha paráfrases legítimas; baixo demais, entrega a resposta de outra pergunta. Calibre com perguntas reais (o histórico do atendimento é o melhor gabarito) e revise as encaminhadas com frequência.
- **Manutenção:** política nova pede FAQ novo. Gere de novo os pares do documento que mudou, aprove e substitua os antigos; senão, a resposta aprovada continua sendo dada depois que a regra mudou.
- **Combinação comum:** respostas prontas na frente, para as perguntas frequentes e sensíveis, e o RAG clássico como plano B para o que ficar abaixo do limiar, com a resposta marcada como gerada, e não aprovada.

### Fala sugerida

> “Aqui a mesma pergunta dá sempre a mesma resposta, porque ela foi escrita e aprovada antes de alguém perguntar.”

---

# Quadro final — o que cada execução provou?

| Execução | O que vimos | Decisão arquitetural |
|---|---|---|
| 1 | O LLM escreve de 4 a 6 pares por política, cada um com o trecho de origem | O LLM trabalha na ingestão, uma vez por versão do documento |
| 2 | Os pares aprovados ficam num arquivo, com `"aprovada": true` | A curadoria humana é o artefato de auditoria |
| 3 | Só as perguntas aprovadas viram vetor | Mesmo modelo e mesma dimensão de embedding na ingestão e na consulta |
| 4 | A paráfrase acima do limiar recebe a resposta pronta; o resto é encaminhado | O limiar decide entre responder e encaminhar: calibre com perguntas reais |
| 5 | Um embedding por pergunta, nenhum token de geração e sempre a mesma resposta | Previsibilidade e custo baixo, em troca de flexibilidade |

## Frase de fechamento

> **Cada variante troca flexibilidade por custo ou previsibilidade. Acervo grande, volátil ou com permissão por usuário pede o RAG clássico.**

---

# Referências oficiais

- Azure OpenAI — structured outputs, o formato fixo dos pares (`responses.parse`):  
  https://learn.microsoft.com/azure/foundry/openai/how-to/structured-outputs

- Azure OpenAI — gerar embeddings:  
  https://learn.microsoft.com/azure/foundry/openai/how-to/embeddings

- Azure AI Search — busca vetorial, onde as perguntas aprovadas morariam em produção:  
  https://learn.microsoft.com/azure/search/vector-search-overview

- Azure AI Search — o score de uma consulta vetorial e como convertê-lo em cosseno:  
  https://learn.microsoft.com/azure/search/vector-search-ranking